# RoamAI — Embed 15 Kauai activitiesRun this in Google Colab. It will:1. Load sentence-transformers/all-MiniLM-L6-v22. Embed each of the 15 Kauai activity descriptions3. Print UPDATE SQL statements to paste into Databricks SQL Editor**Prerequisites:** You've already run `sql/08_seed_kauai_activities.sql`in Databricks SQL Editor, so the 15 activity rows exist in Lakebasewith `description_embedding = NULL`.

## 1. Install sentence-transformers

In [ ]:
!pip install sentence-transformers --quiet

## 2. Load the embedding model(~90 MB download first time, ~30 sec)

In [ ]:
from sentence_transformers import SentenceTransformer

model = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2")
print(f"Model loaded. Embedding dimension: {model.get_sentence_embedding_dimension()}")

## 3. Activity dataDescriptions match `sql/08_seed_kauai_activities.sql` exactly so the UPDATEs match by name.

In [ ]:
activities = [
    ("Kalalau Trail",
     "An 11-mile coastal hiking trail along the rugged Napali Coast on Kauai's north shore. "
     "Considered one of the most spectacular and challenging day hikes in the world, with "
     "dramatic sea cliffs, tropical valleys, hidden waterfalls, and secluded beaches. "
     "Requires permits for the full trail; the first 2 miles to Hanakapiai Beach are open to day hikers."),

    ("Poipu Beach",
     "A crescent-shaped beach on Kauai's sunny south shore, consistently ranked among the best "
     "beaches in the United States. Ideal for swimming, snorkeling, and spotting endangered "
     "Hawaiian monk seals. Two protected coves separated by a sandbar, one calm for children "
     "and one open to the ocean for stronger swimmers."),

    ("Waimea Canyon",
     "Known as the Grand Canyon of the Pacific, Waimea Canyon is 10 miles long, one mile wide, "
     "and over 3,600 feet deep, with dramatic red and green cliffs, cascading waterfalls, and "
     "sweeping views. Multiple lookout points along Waimea Canyon Drive offer stunning photo "
     "opportunities. Best visited in morning for clearest views before afternoon clouds roll in."),

    ("Napali Coast Boat Tour",
     "A guided catamaran or Zodiac raft tour along the Napali Coast, viewing towering sea cliffs, "
     "hidden waterfalls, sea caves, and marine wildlife including dolphins, sea turtles, and "
     "seasonal humpback whales. Available only from May through October when ocean conditions "
     "are calm. Most tours include snorkeling stops."),

    ("Wailua River Kayak",
     "A gentle kayak journey up the Wailua River, Hawaii's only navigable river, through lush "
     "jungle to the Secret Falls hiking trail. The paddle is peaceful and suitable for beginners, "
     "followed by a short hike through the rainforest to a 100-foot waterfall with a swimming "
     "pool at its base. Perfect for calm, sunny days."),

    ("Hanalei Bay",
     "A two-mile-long crescent bay on Kauai's north shore, framed by lush green mountains with "
     "cascading waterfalls after rain. Excellent for surfing in winter, swimming and "
     "paddleboarding in summer. The charming town of Hanalei offers restaurants, shops, and a "
     "historic pier ideal for sunset walks."),

    ("Sleeping Giant Trail",
     "A moderate 3.4-mile round-trip hike up the Nounou Mountain range on Kauai's east side, "
     "ending at panoramic views of coastline, mountains, and Wailua River valley. Named for the "
     "mountain's silhouette resembling a reclining giant. Well-shaded through the forest "
     "sections; open ridge at the top can be windy."),

    ("Anini Beach",
     "A quiet, family-friendly beach on Kauai's north shore, protected by the longest fringing "
     "reef in the Hawaiian Islands. Calm, shallow waters make it one of the safest beaches for "
     "young children and beginner snorkelers. Grassy area with picnic tables and few crowds "
     "even in high season."),

    ("Spouting Horn",
     "A natural blowhole on Kauai's south shore where ocean waves surge through a lava tube "
     "and shoot up to 50 feet into the air, accompanied by a distinctive hissing sound. Free "
     "to view from a cliffside overlook. Best at high tide with strong surf; adjacent lawn "
     "area good for picnics with ocean views."),

    ("Limahuli Garden",
     "A National Tropical Botanical Garden on Kauai's remote north shore, showcasing native "
     "Hawaiian plants across terraced garden levels backed by dramatic Makana mountain. "
     "Self-guided or docent-led tours through ancient Hawaiian agricultural terraces still "
     "growing traditional taro. Small crowds, deep cultural and ecological significance."),

    ("Wailua Falls",
     "An 80-foot twin waterfall on Kauai's east side, easily viewed from a roadside lookout "
     "with no hiking required. Made famous as the opening scene of the Fantasy Island TV series. "
     "Best in morning light. A short unofficial trail leads to the base but is steep and "
     "slippery — not recommended after rain."),

    ("Kokee State Park",
     "A 4,345-acre park at 3,600 feet elevation on Kauai's west side, featuring cool mountain "
     "air, hiking trails through native forests, and spectacular viewpoints overlooking Waimea "
     "Canyon and the remote Kalalau Valley. Home to native honeycreeper birds and the Kokee "
     "Natural History Museum. Cooler temperatures — bring a light jacket."),

    ("Kauai Coffee Company Tour",
     "A free self-guided walking tour through the largest coffee farm in the United States, "
     "with 3,100 acres of coffee trees on Kauai's south shore. Sample multiple coffee varieties "
     "in the visitor center, watch a short film on Hawaiian coffee production, and browse the "
     "gift shop. Fully covered facilities — a good rainy-day activity."),

    ("Kauai Museum",
     "A cultural museum in downtown Lihue showcasing Kauai's history from Polynesian settlement "
     "through the plantation era and modern day. Rotating exhibits on Hawaiian art, geology, "
     "and cultural artifacts. Small but well-curated; a good indoor option on rainy afternoons "
     "or when planning a break from beach and hiking activities."),

    ("Duke's Kauai",
     "A waterfront restaurant on Kalapaki Beach in Lihue, named after Hawaiian surfing legend "
     "Duke Kahanamoku. Menu features fresh local fish, tropical cocktails, and the famous hula "
     "pie for dessert. Open-air lanai seating with ocean views, though the interior offers a "
     "covered option in bad weather. Live Hawaiian music most evenings."),
]

print(f"Loaded {len(activities)} activities to embed")

## 4. Embed all activities (batch encode)

In [ ]:
descriptions = [desc for _, desc in activities]
vectors = model.encode(descriptions, show_progress_bar=True).tolist()

print(f"\nGenerated {len(vectors)} embeddings of dimension {len(vectors[0])}")

## 5. Generate UPDATE SQL — copy the output into Databricks SQL Editor

In [ ]:
print("=" * 70)
print("COPY EVERYTHING BELOW AND PASTE INTO DATABRICKS SQL EDITOR")
print("=" * 70)
print()

for (name, _), vector in zip(activities, vectors):
    vector_str = "[" + ",".join(str(x) for x in vector) + "]"
    safe_name = name.replace("'", "''")
    sql = f"UPDATE activities SET description_embedding = '{vector_str}'::vector WHERE name = '{safe_name}';"
    print(sql)

print()
print("=" * 70)
print(f"Total: {len(activities)} UPDATE statements above.")
print("=" * 70)

## 6. (Optional) Semantic search test queryChange `test_query` to any natural-language question and re-run to seehow the embeddings rank different activities.

In [ ]:
test_query = "peaceful outdoor time near water"
print(f"Test query: {test_query!r}\n")

query_vector = model.encode(test_query).tolist()
query_str = "[" + ",".join(str(x) for x in query_vector) + "]"

test_sql = f"""
SELECT
    name,
    category,
    weather_sensitive,
    LEFT(description, 60) AS preview,
    ROUND((1 - (description_embedding <=> '{query_str}'::vector))::numeric, 3) AS similarity
FROM activities
WHERE description_embedding IS NOT NULL
ORDER BY description_embedding <=> '{query_str}'::vector
LIMIT 5;
"""

print(test_sql)